In [ ]:
!pip install -q -U bitsandbytes peft accelerate datasets transformers scipy

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import os
from tqdm import tqdm
import gc

# ======================================================
# IMPOSTAZIONE FONDAMENTALE: SCEGLI QUALE VERSIONE ADDESTRARE
# ======================================================
VERSION = "v2_2" 

# Setup dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Device attivo: {device}")
print(f" VERSIONE SELEZIONATA: {VERSION}")

# Nomi dei file dinamici in base alla versione scelta
if VERSION == "v2_1":
    INPUT_FILE = "/kaggle/input/datasets/silvioliparoti/dpo-dataset-filtered-v2-fin/dpo_dataset_FILTERED_v2_1.jsonl"
    OUTPUT_DIR = "Zephyr_Manual_DPO_Adapter_v2_1"
    ZIP_NAME = "Zephyr_DPO_FINALE_TESI_v2_1.zip"
    CURVE_NAME = "training_curve_v2_1.png"
    DATA_NAME = "training_loss_data_v2_1.json"
elif VERSION == "v2_2":
    INPUT_FILE = "/kaggle/input/datasets/silvioliparoti/dpo-dataset-filtered-fin/dpo_dataset_FILTERED_v2_2.jsonl"
    OUTPUT_DIR = "Zephyr_Manual_DPO_Adapter_v2_2"
    ZIP_NAME = "Zephyr_DPO_FINALE_TESI_v2_2.zip"
    CURVE_NAME = "training_curve_v2_2.png"
    DATA_NAME = "training_loss_data_v2_2.json"
else:
    raise ValueError("Versione non valida! Scegli 'v2_1' o 'v2_2'")

In [ ]:
# 1. CERCA IL DATASET
dataset_path = None
search_dirs = [".", "/kaggle/working"] + [x[0] for x in os.walk("/kaggle/input")]

for path in search_dirs:
    full_path = os.path.join(path, INPUT_FILE)
    if os.path.exists(full_path):
        dataset_path = full_path
        break

if not dataset_path:
    raise FileNotFoundError(f" Non trovo il file {INPUT_FILE}.")

print(f" Dataset trovato: {dataset_path}")
dataset = load_dataset("json", data_files=dataset_path, split="train")

# 2. CARICAMENTO MODELLO BASE
MODEL_ID = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Carico Zephyr in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Modello e Tokenizer caricati.")

In [ ]:
# 3. APPLICAZIONE LORA

peft_config = LoraConfig(
    r=16,                # Alzato da 16 a 32 (Più capacità di imparare sfumature)
    lora_alpha=32,       # Di regola deve essere il doppio di 'r'
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] 
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Funzione per formattare una coppia
def process_data(examples):
    encoded_chosen = tokenizer(examples['chosen'], truncation=True, max_length=512, padding="max_length")
    encoded_rejected = tokenizer(examples['rejected'], truncation=True, max_length=512, padding="max_length")
    
    return {
        'chosen_input_ids': encoded_chosen['input_ids'],
        'chosen_attention_mask': encoded_chosen['attention_mask'],
        'rejected_input_ids': encoded_rejected['input_ids'],
        'rejected_attention_mask': encoded_rejected['attention_mask'],
    }

print("\n Tokenizzazione del dataset in corso...")
tokenized_dataset = dataset.map(process_data, batched=True, remove_columns=dataset.column_names)
tokenized_dataset.set_format("torch")

train_loader = DataLoader(tokenized_dataset, batch_size=1, shuffle=True)
print(" DataLoader pronto.")

In [ ]:
import torch.nn.functional as F

def get_log_probs(logits, labels):
    labels = labels[:, 1:].clone()
    logits = logits[:, :-1, :]
    padding_mask = (labels != tokenizer.pad_token_id)
    log_probs = F.log_softmax(logits, dim=-1)
    gathered_log_probs = torch.gather(log_probs, -1, labels.unsqueeze(-1)).squeeze(-1)
    return (gathered_log_probs * padding_mask).sum(dim=-1)

def dpo_loss_fn(policy_chosen_logps, policy_rejected_logps, ref_chosen_logps, ref_rejected_logps, beta=0.1):
    pi_logratios = policy_chosen_logps - policy_rejected_logps
    ref_logratios = ref_chosen_logps - ref_rejected_logps
    logits = pi_logratios - ref_logratios
    losses = -F.logsigmoid(beta * logits)
    return losses.mean()

In [ ]:
from tqdm import tqdm

EPOCHS = 1
BETA = 0.1
LR = 1e-5
ACCUMULATION_STEPS = 8 

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
model.train()

loss_history = []
step_history = []

print(f" INIZIO TRAINING MANUALE DPO PER {VERSION} ")

global_step = 0

for epoch in range(EPOCHS):
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader))

    for step, batch in progress_bar:
        c_ids = batch['chosen_input_ids'].to(device)
        c_mask = batch['chosen_attention_mask'].to(device)
        r_ids = batch['rejected_input_ids'].to(device)
        r_mask = batch['rejected_attention_mask'].to(device)
        
        # 1. Forward POLICY
        policy_chosen_logits = model(c_ids, attention_mask=c_mask).logits
        policy_rejected_logits = model(r_ids, attention_mask=r_mask).logits
        policy_chosen_logps = get_log_probs(policy_chosen_logits, c_ids)
        policy_rejected_logps = get_log_probs(policy_rejected_logits, r_ids)
        
        # 2. Forward REFERENCE
        with model.disable_adapter():
            with torch.no_grad():
                ref_chosen_logits = model(c_ids, attention_mask=c_mask).logits
                ref_rejected_logits = model(r_ids, attention_mask=r_mask).logits
                
                ref_chosen_logps = get_log_probs(ref_chosen_logits, c_ids)
                ref_rejected_logps = get_log_probs(ref_rejected_logits, r_ids)
        loss = dpo_loss_fn(
            policy_chosen_logps, policy_rejected_logps,
            ref_chosen_logps, ref_rejected_logps,
            beta=BETA
        )
        # 3. LOSS
        loss_val = loss.item() 
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        if (step + 1) % ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            
            loss_history.append(loss_val)
            step_history.append(global_step)
            
            if global_step % 5 == 0:
                print(f"Step {global_step} | Loss: {loss_val:.4f}")
                
        total_loss += loss_val
        progress_bar.set_description(f"Processing batch {step}")

    print(f" Epoch {epoch+1} Completata.")

print(" TRAINING COMPLETATO!")

In [ ]:
import json
import matplotlib.pyplot as plt

print(" Salvataggio dati e grafici...")

with open(DATA_NAME, "w") as f:
    json.dump({"steps": step_history, "loss": loss_history}, f)

plt.figure(figsize=(10, 6))
plt.plot(step_history, loss_history, label=f"DPO Loss ({VERSION})")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title(f"DPO Training Curve - {VERSION}")
plt.legend()
plt.grid(True)
plt.savefig(CURVE_NAME)

print(f" Grafico salvato come '{CURVE_NAME}'!")
print(f" Dati salvati in '{DATA_NAME}'!")

In [ ]:

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f" Modello salvato con successo nella cartella: {OUTPUT_DIR}")

In [ ]:
import os
from IPython.display import FileLink, display

folder_to_zip = f"/kaggle/working/{OUTPUT_DIR}"

print(f"Zippando i file in {ZIP_NAME}...")

!zip -r {ZIP_NAME} {folder_to_zip} {CURVE_NAME} {DATA_NAME}

print("-" * 40)
print(f" FATTO! File pronto: {ZIP_NAME}")
print("-" * 40)

display(FileLink(ZIP_NAME))